<a href="https://colab.research.google.com/github/AminHasibul/Continual-Learning-UsingInterClusterDistance/blob/main/Simple_Car_V1_unbounded.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
project_dir = '/content/drive/MyDrive/CAR_v1_reproduce'
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)
print(f"Working in: {os.getcwd()}")

Mounted at /content/drive
Working in: /content/drive/MyDrive/CAR_v1_reproduce


In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

Sat Jun 27 22:23:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Then run from that directory
%cd /content/drive/MyDrive/CAR_v1_reproduce
!python car_reproduce.py

/content
Device : cuda
Seeds  : [42, 123, 456]
Results: ./car_results
Loading CIFAR-10...
100% 170M/170M [00:19<00:00, 8.57MB/s]
Task splits : [[0, 1], [2, 3], [4, 5], [6, 7], [8, 9]]
Lambda sweep: [0.1, 1.0, 5.0, 10.0, 20.0, 50.0]
  No checkpoints found — starting fresh.

████████████████████████████████████████████████████████████
  PHASE 1 — LAMBDA SWEEP
  λ values: [0.1, 1.0, 5.0, 10.0, 20.0, 50.0]
  Each λ: 3 seeds × 5 tasks × 20 epochs
████████████████████████████████████████████████████████████

  ───────────────────────────────────────────────────────
  Variant : sweep_lam0.1
  λ       : 0.1
  ───────────────────────────────────────────────────────
    Seed 42 → running...
      Task 1/5  classes=[0, 1]
        epoch 5/20 loss=0.0602
        epoch 10/20 loss=0.0221
        epoch 15/20 loss=0.0155
        epoch 20/20 loss=0.0059
      → acc after task 1: [98.1]
      Task 2/5  classes=[2, 3]
        epoch 5/20 loss=0.1748
        epoch 10/20 loss=-0.0450
        epoch 15/20 loss

car_debug.py
============
Verification script for car_reproduce.py
Runs ONE seed (42), ONE lambda (0.1), full 5-task sequence
with all 5 diagnostic checks printed at each tas

In [ ]:
"""
car_debug.py
============
Verification script for car_reproduce.py
Runs ONE seed (42), ONE lambda (0.1), full 5-task sequence
with all 5 diagnostic checks printed at each task.

Does NOT save checkpoints or overwrite any results.
Run AFTER car_reproduce.py has already completed.

Usage:
  !python car_debug.py

Expected output documented inline below each check.
"""

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, ConcatDataset
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18

# ─────────────────────────────────────────────
# CONFIG — fixed for debug run
# ─────────────────────────────────────────────
SEED            = 42
ICF_LAMBDA      = 0.1
NUM_TASKS       = 5
EPOCHS_PER_TASK = 20
BATCH_SIZE      = 32
LR              = 0.001
EXEMPLARS_PER_CLASS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# ─────────────────────────────────────────────
# DATA
# ─────────────────────────────────────────────
class ClassSubset(Dataset):
    def __init__(self, base_dataset, classes):
        self.base    = base_dataset
        targets      = np.array(base_dataset.targets)
        self.indices = np.where(np.isin(targets, classes))[0]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        return self.base[self.indices[idx]]

class ListDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        x, y = self.items[i]
        return x, y


# ─────────────────────────────────────────────
# MODEL
# ─────────────────────────────────────────────
def build_model():
    model         = resnet18(weights=None, num_classes=10)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3,
                               stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model.to(DEVICE)

def get_features(model, x):
    x = model.conv1(x);   x = model.bn1(x)
    x = model.relu(x);    x = model.maxpool(x)
    x = model.layer1(x);  x = model.layer2(x)
    x = model.layer3(x);  x = model.layer4(x)
    x = model.avgpool(x)
    return torch.flatten(x, 1)


# ─────────────────────────────────────────────
# REPLAY BUFFER
# ─────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, exemplars_per_class=20):
        self.n    = exemplars_per_class
        self.data = {}

    def add(self, dataset, classes):
        targets = np.array(dataset.targets)
        for c in classes:
            idx    = np.where(targets == c)[0]
            chosen = np.random.choice(
                idx, size=min(self.n, len(idx)), replace=False)
            self.data[c] = [(dataset[i][0], dataset[i][1])
                            for i in chosen]

    def as_dataset(self):
        items = []
        for v in self.data.values():
            items.extend(v)
        return items

    def __len__(self):
        return sum(len(v) for v in self.data.values())


# ─────────────────────────────────────────────
# ICF LOSS
# ─────────────────────────────────────────────
class ICFLoss(nn.Module):
    def __init__(self, lam):
        super().__init__()
        self.lam       = lam
        self.centroids = {}

    @torch.no_grad()
    def update_centroids(self, model, dataset, classes):
        model.eval()
        loader          = DataLoader(dataset, batch_size=256, shuffle=False)
        feats_per_class = {c: [] for c in classes}
        for x, y in loader:
            x = x.to(DEVICE)
            f = F.normalize(get_features(model, x), p=2, dim=1)
            for c in classes:
                mask = (y == c)
                if mask.any():
                    feats_per_class[c].append(f[mask].cpu())
        for c in classes:
            if feats_per_class[c]:
                cat = torch.cat(feats_per_class[c], dim=0)
                mu  = cat.mean(0)
                self.centroids[c] = F.normalize(mu, p=2, dim=0).to(DEVICE)
        model.train()

    def forward(self, features):
        if not self.centroids:
            return torch.tensor(0.0, device=features.device)
        f_norm = F.normalize(features, p=2, dim=1)
        loss   = torch.tensor(0.0, device=features.device)
        for mu in self.centroids.values():
            dist  = torch.norm(f_norm - mu.unsqueeze(0), dim=1)
            loss += -dist.mean()
        loss /= len(self.centroids)
        return self.lam * loss


# ─────────────────────────────────────────────
# EVAL
# ─────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, dataset, classes):
    subset = ClassSubset(dataset, classes)
    loader = DataLoader(subset, batch_size=256, shuffle=False)
    correct, total = 0, 0
    model.eval()
    for x, y in loader:
        x, y    = x.to(DEVICE), y.to(DEVICE)
        preds   = model(x).argmax(dim=1)
        correct += (preds == y).sum().item()
        total   += y.size(0)
    return correct / total if total > 0 else 0.0


# ─────────────────────────────────────────────
# SEPARATOR HELPER
# ─────────────────────────────────────────────
def sep(title=""):
    print(f"\n{'─'*55}")
    if title:
        print(f"  {title}")
        print(f"{'─'*55}")


# ─────────────────────────────────────────────
# MAIN DEBUG RUN
# ─────────────────────────────────────────────
def main():
    print("="*55)
    print("  CAR DEBUG VERIFICATION")
    print(f"  Seed={SEED}  λ={ICF_LAMBDA}  device={DEVICE}")
    print("="*55)

    set_seed(SEED)

    print("\nLoading CIFAR-10...")
    train_ds = torchvision.datasets.CIFAR10(
        root='./data', train=True, download=True, transform=TRANSFORM)
    test_ds  = torchvision.datasets.CIFAR10(
        root='./data', train=False, download=True, transform=TRANSFORM)

    task_classes = [list(range(i*2, i*2+2)) for i in range(NUM_TASKS)]

    model     = build_model()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.CrossEntropyLoss()
    buffer    = ReplayBuffer(EXEMPLARS_PER_CLASS)
    icf       = ICFLoss(lam=ICF_LAMBDA)

    acc_matrix = np.full((NUM_TASKS, NUM_TASKS), -1.0)

    for t, classes in enumerate(task_classes):

        print(f"\n{'█'*55}")
        print(f"  TASK {t+1}/5   classes={classes}")
        print(f"{'█'*55}")

        # ── Build datasets ────────────────────────────────────
        current_ds = ClassSubset(train_ds, classes)

        if len(buffer) > 0:
            replay_ds = ListDataset(buffer.as_dataset())
            combined  = ConcatDataset([current_ds, replay_ds])
        else:
            replay_ds = None
            combined  = current_ds

        # ══════════════════════════════════════════════════════
        # CHECK 2 — Dataset sizes
        # Expected Task 1: current≈10000, replay=0,  combined≈10000
        # Expected Task 2: current≈10000, replay=40, combined≈10040
        # Expected Task 3: current≈10000, replay=80, combined≈10080
        # ══════════════════════════════════════════════════════
        sep("CHECK 2 — Dataset sizes")
        print(f"  Current task size : {len(current_ds)}")
        print(f"  Replay size       : "
              f"{len(replay_ds) if replay_ds else 0}")
        print(f"  Combined size     : {len(combined)}")

        expected_replay = t * EXEMPLARS_PER_CLASS * len(classes)
        status = "✓" if len(buffer) == expected_replay else "✗ MISMATCH"
        print(f"  Expected replay   : {expected_replay}  {status}")

        loader = DataLoader(combined, batch_size=BATCH_SIZE,
                            shuffle=True, num_workers=2,
                            pin_memory=True)

        # ══════════════════════════════════════════════════════
        # CHECK 5 — Batch labels (first batch only)
        # Expected Task 1: only [0,1]
        # Expected Task 2: [0,1] from replay + [2,3] current
        # Expected Task 3: [0,1,2,3] from replay + [4,5] current
        # ══════════════════════════════════════════════════════
        sep("CHECK 5 — Batch labels (first batch)")
        for x_b, y_b in loader:
            batch_labels = sorted(set(y_b.tolist()))
            print(f"  Labels in batch : {batch_labels}")
            expected_labels = sorted(
                classes +
                [c for c in buffer.data.keys()])
            print(f"  Expected labels : {expected_labels}")
            match = set(batch_labels).issubset(set(expected_labels))
            print(f"  Subset check    : {'✓' if match else '✗ UNEXPECTED LABELS'}")
            break

        # ── Train with CE+ICF loss, print first epoch ─────────
        print(f"\n  Training {EPOCHS_PER_TASK} epochs...")
        for ep in range(EPOCHS_PER_TASK):
            model.train()
            epoch_loss = 0.0
            ce_samples, icf_samples = [], []

            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                optimizer.zero_grad()
                feats  = get_features(model, x)
                logits = model.fc(feats)

                ce_val  = criterion(logits, y)
                icf_val = icf(feats)
                loss    = ce_val + icf_val

                loss.backward()
                optimizer.step()
                epoch_loss   += loss.item()
                ce_samples.append(ce_val.item())
                icf_samples.append(icf_val.item())

            avg_loss = epoch_loss / len(loader)
            avg_ce   = np.mean(ce_samples)
            avg_icf  = np.mean(icf_samples)

            # ══════════════════════════════════════════════════
            # CHECK 4 — CE vs ICF loss magnitudes
            # Expected Task 1: ICF=0.0 (no centroids yet)
            # Expected Task 2+: ICF should be small negative at λ=0.1
            # Red flag: |ICF| >> CE means ICF dominating
            # ══════════════════════════════════════════════════
            if ep == 0 or (ep + 1) % 5 == 0:
                domination = abs(avg_icf) > abs(avg_ce)
                flag = "⚠ ICF DOMINATING" if domination else "✓ balanced"
                print(f"  Epoch {ep+1:>2}/{EPOCHS_PER_TASK}  "
                      f"total={avg_loss:.4f}  "
                      f"CE={avg_ce:.4f}  "
                      f"ICF={avg_icf:.4f}  {flag}")

        # ── Update ICF centroids ──────────────────────────────
        icf.update_centroids(
            model, ClassSubset(train_ds, classes), classes)

        # ══════════════════════════════════════════════════════
        # CHECK 3 — Centroid count after update
        # Expected after Task 1: 2 centroids  (classes 0,1)
        # Expected after Task 2: 4 centroids  (classes 0,1,2,3)
        # Expected after Task 3: 6 centroids
        # Expected after Task 4: 8 centroids
        # Expected after Task 5: 10 centroids
        # ══════════════════════════════════════════════════════
        sep("CHECK 3 — Centroid state after update")
        print(f"  Centroids stored : {sorted(icf.centroids.keys())}")
        print(f"  Num centroids    : {len(icf.centroids)}")
        expected_centroids = (t + 1) * len(classes)
        status = "✓" if len(icf.centroids) == expected_centroids else "✗ MISMATCH"
        print(f"  Expected         : {expected_centroids}  {status}")

        # Check centroid norms (should all be 1.0 after L2 norm)
        norms = [icf.centroids[c].norm().item()
                 for c in icf.centroids]
        print(f"  Centroid norms   : "
              f"min={min(norms):.4f}  max={max(norms):.4f}  "
              f"(expected ~1.0)")

        # ── Update replay buffer ──────────────────────────────
        buffer.add(train_ds, classes)

        # ══════════════════════════════════════════════════════
        # CHECK 1 — Replay buffer state
        # Expected after Task 1: total=40,  classes=[0,1],  20 each
        # Expected after Task 2: total=80,  classes=[0,1,2,3]
        # Expected after Task 3: total=120, classes=[0..5]
        # Expected after Task 4: total=160, classes=[0..7]
        # Expected after Task 5: total=200, classes=[0..9]
        # ══════════════════════════════════════════════════════
        sep("CHECK 1 — Replay buffer state")
        print(f"  Buffer classes   : {sorted(buffer.data.keys())}")
        print(f"  Buffer total     : {len(buffer)}")
        expected_total = (t + 1) * EXEMPLARS_PER_CLASS * len(classes)
        status = "✓" if len(buffer) == expected_total else "✗ MISMATCH"
        print(f"  Expected total   : {expected_total}  {status}")
        print(f"  Per-class counts :")
        for c in sorted(buffer.data.keys()):
            n      = len(buffer.data[c])
            ok     = "✓" if n == EXEMPLARS_PER_CLASS else f"✗ expected {EXEMPLARS_PER_CLASS}"
            print(f"    class {c}: {n} exemplars  {ok}")

        # ── Evaluate on all seen tasks ────────────────────────
        sep("Accuracy after this task")
        for j in range(t + 1):
            acc = evaluate(model, test_ds, task_classes[j])
            acc_matrix[t][j] = acc
            print(f"  Task {j+1} (classes {task_classes[j]}): "
                  f"{acc*100:.1f}%")

    # ── Final summary ─────────────────────────────────────────
    print("\n" + "="*55)
    print("  FINAL ACCURACY MATRIX")
    print("  (matches Table I if all checks passed)")
    print("="*55)
    print("After\\Task  " + "  ".join(
        [f"  T{j+1}  " for j in range(NUM_TASKS)]))
    print("─"*55)
    for i in range(NUM_TASKS):
        row = f"  Task {i+1}    "
        for j in range(NUM_TASKS):
            if acc_matrix[i][j] < 0:
                row += "   –   "
            else:
                row += f" {acc_matrix[i][j]*100:4.1f}  "
        print(row)

    # ── Check against reproduction results ───────────────────
    print("\n" + "="*55)
    print("  CROSS-CHECK vs raw_numbers.txt (seed 42)")
    print("="*55)
    expected_seed42 = {
        (0,0): 98.1,
        (1,0): 25.4, (1,1): 87.7,
        (2,0):  6.3, (2,1):  0.3, (2,2): 91.8,
        (3,0):  9.6, (3,1):  0.9, (3,2):  1.9, (3,3): 96.2,
        (4,0):  0.4, (4,1):  0.5, (4,2): 17.3, (4,3):  6.8, (4,4): 96.5,
    }
    all_match = True
    for (i,j), expected in expected_seed42.items():
        got   = acc_matrix[i][j] * 100
        diff  = abs(got - expected)
        ok    = "✓" if diff < 0.2 else f"✗ diff={diff:.1f}pp"
        if diff >= 0.2:
            all_match = False
        print(f"  T{i+1} after task {j+1}: "
              f"got={got:.1f}%  expected={expected:.1f}%  {ok}")

    print("\n" + ("✓ All checks passed — reproduction is consistent"
                  if all_match else
                  "✗ Some values differ — investigate above"))
    print("="*55)


if __name__ == "__main__":
    main()

  CAR DEBUG VERIFICATION
  Seed=42  λ=0.1  device=cuda

Loading CIFAR-10...

███████████████████████████████████████████████████████
  TASK 1/5   classes=[0, 1]
███████████████████████████████████████████████████████

───────────────────────────────────────────────────────
  CHECK 2 — Dataset sizes
───────────────────────────────────────────────────────
  Current task size : 10000
  Replay size       : 0
  Combined size     : 10000
  Expected replay   : 0  ✓

───────────────────────────────────────────────────────
  CHECK 5 — Batch labels (first batch)
───────────────────────────────────────────────────────
  Labels in batch : [0, 1]
  Expected labels : [0, 1]
  Subset check    : ✓

  Training 20 epochs...
  Epoch  1/20  total=0.3184  CE=0.3184  ICF=0.0000  ✓ balanced
  Epoch  5/20  total=0.0675  CE=0.0675  ICF=0.0000  ✓ balanced
  Epoch 10/20  total=0.0301  CE=0.0301  ICF=0.0000  ✓ balanced
  Epoch 15/20  total=0.0133  CE=0.0133  ICF=0.0000  ✓ balanced
  Epoch 20/20  total=0.0095  CE=

In [ ]:
# ============================================================
# CIFAR-100 ICF CONTROL — does ICF contribute on a 2nd dataset?
# ============================================================
# Closes the gap: you only ran the omega=0 control on CIFAR-10.
# Holds NCM+distill+buffer constant, toggles ONLY omega.
#
# MATCHES your validated CIFAR-100 protocol:
#   10 tasks, exemplars_per_class=10, epochs_per_task=10, distill ON, NCM ON
#   Reference (omega=0.5, validated): CAR-Enhanced = 40.45 +/- 0.20
#
# omega=0 ~= 40.45  -> ICF inert on CIFAR-100 too (replicates CIFAR-10)
# omega=0 <<  40.45 -> ICF DOES contribute here (dataset-dependent!)
#
# 3 seeds x ~33 min = ~1.7h. Resumable. Set SEEDS=[42] for a fast read.
# ============================================================

import os, sys, json, time, importlib
import numpy as np

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

CODE_DIR = '/content/drive/MyDrive/AML'
OUT_DIR  = '/content/drive/MyDrive/AML/paper_v3/experiments'
sys.path.insert(0, CODE_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

import car_adaptive_icf
importlib.reload(car_adaptive_icf)
from car_adaptive_icf import get_split_cifar, run_experiment, CAR_Static

try:
    from IPython.display import display, Javascript
    display(Javascript('''function _k(){document.querySelector(
      "#top-toolbar > colab-connect-button")?.click();} setInterval(_k,60000);'''))
except Exception:
    pass

SEEDS = [42, 123, 456]
EPOCHS = 10          # CIFAR-100 validated setting
BUFFER = 10          # CIFAR-100 validated setting (CAR-Enhanced used buf=10)

print("Loading cifar100 (10 tasks)...")
tr, te, tc, ncls = get_split_cifar(dataset='cifar100', num_tasks=10)

out_path = f'{OUT_DIR}/icf_off_cifar100.json'
results = {}
if os.path.exists(out_path):
    with open(out_path) as f: results = json.load(f)
    print(f"Resuming — {len(results)} seed(s) done")

for seed in SEEDS:
    key = f'seed_{seed}'
    if key in results:
        print(f"  skip {key} (acc={results[key]['accuracy']:.2f}%)"); continue
    print(f"\n--- CIFAR-100 ICF-off (omega=0) seed {seed} ---")
    t0 = time.time()
    r = run_experiment(
        CAR_Static, 'NCM+Distill-only (ICF off)',
        {'omega': 0.0, 'exemplars_per_class': BUFFER,
         'use_distillation': True, 'distill_weight': 1.0,
         'epochs_per_task': EPOCHS},
        tr, te, tc, num_classes=ncls, seed=seed, verbose=False, use_ncm=True)
    print(f"    acc={r['final_avg_accuracy']:.2f}%  forget={r['avg_forgetting']:.2f}%  ({(time.time()-t0)/60:.1f}m)")
    results[key] = {'accuracy': r['final_avg_accuracy'],
                    'forgetting': r['avg_forgetting'],
                    'accuracy_matrix': r['accuracy_matrix']}
    with open(out_path, 'w') as f: json.dump(results, f, indent=2)
    print(f"    saved -> {out_path}")

a = [v['accuracy'] for v in results.values()]
fo = [v['forgetting'] for v in results.values()]
off_a, off_s = np.mean(a), np.std(a)
on_a, on_s = 40.45, 0.20   # your validated CIFAR-100 CAR-Enhanced (omega=0.5)

print(f"\n{'='*62}")
print("DOES ICF CONTRIBUTE ON CIFAR-100?  (NCM+distill+buf10 held constant)")
print(f"{'='*62}")
print(f"  ICF OFF (omega=0)  : {off_a:.2f} ± {off_s:.2f}%   forget {np.mean(fo):.2f}%")
print(f"  ICF ON  (omega=0.5): {on_a:.2f} ± {on_s:.2f}%   (validated CAR-Enhanced)")
print(f"  ICF's contribution : {on_a-off_a:+.2f}%")
print(f"\n  CIFAR-10 reference: ICF contributed +0.72% (inert).")
print("  Similar here -> ICF inert on BOTH datasets (strong claim).")
print("  Very different -> ICF contribution is dataset-dependent (report honestly).")